# EDA: Ventas MercadoLibre — Bronze Layer
**Tabla:** `iniciacion_deportiva.bronze.ventas_raw`  
**Objetivo:** Entender la estructura, calidad y distribución de los datos para planificar la capa Silver.

---
## Contexto del dataset
- Ventas reales del negocio familiar desde MercadoLibre API
- Ordenes históricas cargadas mes a mes: 2025-01-01 → 2026-03-04 (~9600 registros)
- Todo en **STRING** en Bronze 
- `_rescued_data`: Auto Loader la llena si llega un campo fuera de schema
- **Nota de negocio (a verificar):** Un cliente puede tener múltiples `id_venta` con la misma fecha/hora porque MercadoLibre devuelve una orden por producto. En Gold los agruparemos para reconstruir pedidos completos.

## Paso 1: Estructura y Exploración Inicial
*¿Qué tenemos?*

In [0]:
%sql
-- 1.1: ¿Cuántos registros tenemos?
SELECT COUNT(*) AS total_registros
FROM iniciacion_deportiva.bronze.ventas_raw;

In [0]:
%sql
DESCRIBE TABLE iniciacion_deportiva.bronze.ventas_raw

In [0]:
%sql
-- 1.2: Ver las primeras filas sin columnas de metadata
SELECT
    id_venta,
    fecha,
    id_producto,
    titulo_producto,
    cantidad,
    categoria_id,
    precio_unitario,
    comision,
    id_cliente,
    cliente_nickname,
    estado
FROM iniciacion_deportiva.bronze.ventas_raw
LIMIT 20;

In [0]:
%sql
-- ¿Cuántos clientes tienen múltiples id_venta en el mismo minuto?
-- Si el número es alto, el patrón de "pedido multi-ítem" es real
SELECT
    COUNT(*) AS casos
FROM (
    SELECT id_cliente, SUBSTRING(fecha, 1, 16) AS fecha_minuto
    FROM iniciacion_deportiva.bronze.ventas_raw
    GROUP BY id_cliente, SUBSTRING(fecha, 1, 16)
    HAVING COUNT(DISTINCT id_venta) > 1
);

In [0]:
%sql
-- Ver ejemplos concretos: mismo cliente, mismo minuto, productos distintos
-- Esto confirma que son ítems de un mismo pedido conceptual
WITH pedidos_multi AS (
    SELECT id_cliente, SUBSTRING(fecha, 1, 16) AS fecha_minuto
    FROM iniciacion_deportiva.bronze.ventas_raw
    GROUP BY id_cliente, SUBSTRING(fecha, 1, 16)
    HAVING COUNT(DISTINCT id_venta) > 1
    LIMIT 5
)
SELECT
    v.id_cliente,
    v.cliente_nickname,
    SUBSTRING(v.fecha, 1, 16)  AS fecha_minuto,
    v.id_venta,
    v.id_producto,
    v.titulo_producto,
    v.cantidad,
    v.precio_unitario
    FROM iniciacion_deportiva.bronze.ventas_raw v
    INNER JOIN pedidos_multi p
    ON v.id_cliente = p.id_cliente
    AND SUBSTRING(v.fecha, 1, 16) = p.fecha_minuto
ORDER BY v.id_cliente, v.fecha;

In [0]:
%sql
-- 1.3: Cardinalidad — ¿cuántos valores únicos tiene cada columna clave?
SELECT
    COUNT(DISTINCT id_venta)        AS ventas_unicas,
    COUNT(DISTINCT id_producto)     AS productos_unicos,
    COUNT(DISTINCT id_cliente)      AS clientes_unicos,
    COUNT(DISTINCT categoria_id)    AS categorias_unicas,
    COUNT(DISTINCT estado)          AS estados_unicos,
    COUNT(DISTINCT titulo_producto) AS titulos_unicos
FROM iniciacion_deportiva.bronze.ventas_raw;

In [0]:
%sql
-- ¿Cuántos títulos distintos tiene cada id_producto? Check
SELECT
    id_producto,
    COUNT(DISTINCT titulo_producto) AS titulos_distintos,
    COLLECT_SET(titulo_producto)    AS lista_titulos
FROM iniciacion_deportiva.bronze.ventas_raw
GROUP BY id_producto
HAVING COUNT(DISTINCT titulo_producto) > 1
ORDER BY titulos_distintos DESC
LIMIT 20;

In [0]:
%sql
-- Posible Lógica para Silver: título más reciente por producto
SELECT DISTINCT
    id_producto,
    FIRST_VALUE(titulo_producto) OVER (
        PARTITION BY id_producto 
        ORDER BY fecha DESC
    ) AS titulo_producto
FROM iniciacion_deportiva.bronze.ventas_raw 

In [0]:
%sql
-- Forma correcta con ROW_NUMBER
WITH titulos_rankeados AS (
    SELECT
        id_producto,
        titulo_producto,
        ROW_NUMBER() OVER (
            PARTITION BY id_producto
            ORDER BY fecha DESC
        ) AS rn
    FROM ventas_raw
)
SELECT id_producto, titulo_producto
FROM titulos_rankeados
WHERE rn = 1

## Paso 2: Análisis de Nulos
*¿Qué tan completos están los datos?*

In [0]:
%sql
-- 2.1: Conteo absoluto de nulos por columna
WITH total AS (
    SELECT COUNT(*) AS n
    FROM iniciacion_deportiva.bronze.ventas_raw
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(id_venta)         AS nulos_id_venta,
        COUNT(*) - COUNT(fecha)            AS nulos_fecha,
        COUNT(*) - COUNT(id_producto)      AS nulos_id_producto,
        COUNT(*) - COUNT(titulo_producto)  AS nulos_titulo,
        COUNT(*) - COUNT(cantidad)         AS nulos_cantidad,
        COUNT(*) - COUNT(categoria_id)     AS nulos_categoria,
        COUNT(*) - COUNT(precio_unitario)  AS nulos_precio,
        COUNT(*) - COUNT(comision)         AS nulos_comision,
        COUNT(*) - COUNT(id_cliente)       AS nulos_id_cliente,
        COUNT(*) - COUNT(cliente_nickname) AS nulos_nickname,
        COUNT(*) - COUNT(estado)           AS nulos_estado
    FROM iniciacion_deportiva.bronze.ventas_raw
)
SELECT t.n AS total_registros, n.*
FROM total t, nulos n;

In [0]:
%sql
-- 2.2: Porcentaje de nulos en columnas clave
WITH totales AS (
    SELECT COUNT(*) AS total FROM iniciacion_deportiva.bronze.ventas_raw
)
SELECT
    ROUND((t.total - COUNT(id_venta))        * 100.0 / t.total, 2) AS pct_nulos_id_venta,
    ROUND((t.total - COUNT(fecha))           * 100.0 / t.total, 2) AS pct_nulos_fecha,
    ROUND((t.total - COUNT(precio_unitario)) * 100.0 / t.total, 2) AS pct_nulos_precio,
    ROUND((t.total - COUNT(comision))        * 100.0 / t.total, 2) AS pct_nulos_comision,
    ROUND((t.total - COUNT(id_cliente))      * 100.0 / t.total, 2) AS pct_nulos_id_cliente,
    ROUND((t.total - COUNT(estado))          * 100.0 / t.total, 2) AS pct_nulos_estado
FROM iniciacion_deportiva.bronze.ventas_raw
CROSS JOIN totales t
GROUP BY t.total;

In [0]:
%sql
-- 2.3: ¿Hay _rescued_data?
-- Auto Loader la llena cuando un campo llega en formato inesperado (schema mismatch)
-- Si hay valores acá, algo en la API cambió y Bronze lo capturó igual
SELECT
    COUNT(*)                                            AS total_registros,
    COUNT(_rescued_data)                                AS con_rescued_data,
    ROUND(COUNT(_rescued_data) * 100.0 / COUNT(*), 2)  AS pct_rescued
FROM iniciacion_deportiva.bronze.ventas_raw;

## Paso 3: Cardinalidad y Distribución de Categóricos
*¿Qué valores tienen las columnas de texto?*

In [0]:
%sql
-- 3.1: Distribución de estados de venta
-- Clave para entender tasa de cancelación — dato central para Gold
SELECT
    estado,
    COUNT(*)                                           AS cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM iniciacion_deportiva.bronze.ventas_raw
GROUP BY estado
ORDER BY cantidad DESC;

In [0]:
%sql
-- 3.2: Top 20 productos más vendidos (solo órdenes pagadas)
SELECT
    id_producto,
    titulo_producto,
    COUNT(*)                   AS ordenes,
    SUM(CAST(cantidad AS INT)) AS unidades_totales
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE estado = 'paid'
GROUP BY id_producto, titulo_producto
ORDER BY ordenes DESC
LIMIT 20;

In [0]:
%sql
-- 3.3: Distribución por categoría (despues cruzar con nombres de categorias que vienen de otro endpoint)
SELECT
    categoria_id,
    COUNT(*)                                           AS ordenes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM iniciacion_deportiva.bronze.ventas_raw
GROUP BY categoria_id
ORDER BY ordenes DESC;

In [0]:
%sql
-- 3.4: Top 20 clientes por frecuencia de compra
SELECT
    id_cliente,
    cliente_nickname,
    COUNT(*)                   AS ordenes,
    SUM(CAST(cantidad AS INT)) AS unidades_totales
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE estado = 'paid'
GROUP BY id_cliente, cliente_nickname
ORDER BY ordenes DESC
LIMIT 20;

## Paso 4: Rangos y Outliers en Variables Numéricas
*¿Los números tienen sentido?*

> **Importante:** Todo viene como STRING desde Bronze. Casteamos acá solo para analizar.
> En Silver lo hacemos definitivamente con `TRY_CAST` y validaciones.

In [0]:
%sql
-- 4.1: Estadísticas de precio_unitario
SELECT
    COUNT(*)                                                     AS registros,
    MIN(CAST(precio_unitario AS DOUBLE))                         AS precio_min,
    MAX(CAST(precio_unitario AS DOUBLE))                         AS precio_max,
    ROUND(AVG(CAST(precio_unitario AS DOUBLE)), 2)               AS precio_promedio,
    ROUND(PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.5),  2) AS precio_mediana,
    ROUND(PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.25), 2) AS precio_p25,
    ROUND(PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.75), 2) AS precio_p75
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE precio_unitario IS NOT NULL AND precio_unitario != '';

In [0]:
%sql
-- 4.2: Estadísticas de cantidad vendida
SELECT
    COUNT(*)                                   AS registros,
    MIN(CAST(cantidad AS INT))                 AS cantidad_min,
    MAX(CAST(cantidad AS INT))                 AS cantidad_max,
    ROUND(AVG(CAST(cantidad AS DOUBLE)), 2)    AS cantidad_promedio,
    PERCENTILE(CAST(cantidad AS INT), 0.95)    AS cantidad_p95
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE cantidad IS NOT NULL AND cantidad != '';

In [0]:
%sql
-- 4.3: Outliers en precio — ¿hay valores absurdos?
WITH stats AS (
    SELECT
        PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.01) AS p01,
        PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.99) AS p99
    FROM iniciacion_deportiva.bronze.ventas_raw
    WHERE precio_unitario IS NOT NULL AND precio_unitario != ''
)
SELECT 'Muy bajo (< P01)' AS tipo_outlier, COUNT(*) AS cantidad
FROM iniciacion_deportiva.bronze.ventas_raw, stats
WHERE CAST(precio_unitario AS DOUBLE) < stats.p01
UNION ALL
SELECT 'Muy alto (> P99)' AS tipo_outlier, COUNT(*) AS cantidad
FROM iniciacion_deportiva.bronze.ventas_raw, stats
WHERE CAST(precio_unitario AS DOUBLE) > stats.p99;

In [0]:
%sql
-- Verificar los productos con precios extremos
WITH stats AS (
    SELECT
        PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.01) AS p01,
        PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.99) AS p99
    FROM iniciacion_deportiva.bronze.ventas_raw
    WHERE precio_unitario IS NOT NULL AND precio_unitario != ''
)
SELECT 
    v.fecha,
    v.titulo_producto, -- O la columna que identifique el artículo
    CAST(v.precio_unitario AS DOUBLE) AS precio_analizado,
    CASE 
        WHEN CAST(v.precio_unitario AS DOUBLE) < s.p01 THEN 'MUY BAJO (P01)'
        WHEN CAST(v.precio_unitario AS DOUBLE) > s.p99 THEN 'MUY ALTO (P99)'
    END AS tipo_outlier,
    s.p01 AS limite_inferior,
    s.p99 AS limite_superior
FROM iniciacion_deportiva.bronze.ventas_raw v
CROSS JOIN stats s
WHERE 
    (CAST(v.precio_unitario AS DOUBLE) < s.p01 
     OR CAST(v.precio_unitario AS DOUBLE) > s.p99)
    AND v.precio_unitario IS NOT NULL 
    AND v.precio_unitario != ''
ORDER BY precio_analizado DESC;

In [0]:
%sql
-- 4.4: ¿Hay precios en 0 o negativos? (datos claramente inválidos)
SELECT COUNT(*) AS precios_invalidos
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE TRY_CAST(precio_unitario AS DOUBLE) <= 0
   OR precio_unitario IS NULL
   OR precio_unitario = '';

## Paso 5: Duplicados y Calidad General
*¿Los datos son confiables?*

> En Bronze **se esperan duplicados** — el ETL incremental corre cada 15 min con ventana de 2 horas,
> entonces cada venta puede aparecer hasta ~8 veces. Silver los elimina con `MERGE ON id_venta`.

In [0]:
%sql
-- 5.1: ¿Cuántos id_venta están duplicados en Bronze?
SELECT
    COUNT(*)                              AS total_filas,
    COUNT(DISTINCT id_venta)              AS ventas_unicas,
    COUNT(*) - COUNT(DISTINCT id_venta)   AS duplicados_reales
FROM iniciacion_deportiva.bronze.ventas_raw;

In [0]:
%sql
-- 5.2: Ver un ejemplo de duplicado — ¿son idénticos o cambia el estado?
-- Si cambia el estado (ej: paid → cancelled), NO son duplicados reales:
-- son actualizaciones que Silver debe procesar con MERGE
WITH dup_ids AS (
    SELECT id_venta
    FROM iniciacion_deportiva.bronze.ventas_raw
    GROUP BY id_venta
    HAVING COUNT(*) > 1
)
SELECT v.id_venta, v.fecha, v.titulo_producto, v.estado, v.ingested_at
FROM iniciacion_deportiva.bronze.ventas_raw v
INNER JOIN dup_ids d ON v.id_venta = d.id_venta
ORDER BY v.id_venta, v.ingested_at;

In [0]:
%sql
-- 5.3: Reporte consolidado de calidad
WITH total AS (
    SELECT COUNT(*) AS n FROM iniciacion_deportiva.bronze.ventas_raw
),
problemas AS (
    SELECT
        COUNT(CASE WHEN id_venta IS NULL OR id_venta = ''                THEN 1 END) AS sin_id_venta,
        COUNT(CASE WHEN fecha IS NULL OR fecha = ''                      THEN 1 END) AS sin_fecha,
        COUNT(CASE WHEN precio_unitario IS NULL OR precio_unitario = '' THEN 1 END) AS sin_precio,
        COUNT(CASE WHEN TRY_CAST(precio_unitario AS DOUBLE) <= 0         THEN 1 END) AS precio_invalido,
        COUNT(CASE WHEN cantidad IS NULL OR cantidad = ''                THEN 1 END) AS sin_cantidad,
        COUNT(CASE WHEN TRY_CAST(cantidad AS INT) <= 0                   THEN 1 END) AS cantidad_invalida,
        COUNT(CASE WHEN estado IS NULL OR estado = ''                    THEN 1 END) AS sin_estado,
        COUNT(CASE WHEN id_cliente IS NULL OR id_cliente = ''            THEN 1 END) AS sin_cliente
    FROM iniciacion_deportiva.bronze.ventas_raw
)
SELECT
    t.n                                            AS total_registros,
    p.sin_id_venta,
    p.sin_fecha,
    p.sin_precio,
    ROUND(p.sin_precio     * 100.0 / t.n, 2)      AS pct_sin_precio,
    p.precio_invalido,
    p.sin_cantidad,
    p.cantidad_invalida,
    p.sin_estado,
    p.sin_cliente
FROM total t, problemas p;

## Paso 6: Análisis Temporal
*¿Cómo evolucionan las ventas en el tiempo?*

In [0]:
%sql
-- 6.1: Rango temporal del dataset
SELECT
    MIN(SUBSTRING(fecha, 1, 10))                   AS fecha_inicio,
    MAX(SUBSTRING(fecha, 1, 10))                   AS fecha_fin,
    COUNT(DISTINCT SUBSTRING(fecha, 1, 7))         AS meses_con_datos 
FROM iniciacion_deportiva.bronze.ventas_raw;

In [0]:
%sql
-- 6.2: Ventas y tasa de cancelación por mes
-- SUBSTRING(fecha, 1, 7) extrae YYYY-MM del string ISO 8601
SELECT
    SUBSTRING(fecha, 1, 7)                                             AS mes,
    COUNT(*)                                                           AS total_ordenes
FROM iniciacion_deportiva.bronze.ventas_raw
GROUP BY SUBSTRING(fecha, 1, 7)
ORDER BY mes;

In [0]:
SELECT
    DAYOFWEEK(CAST(fecha AS TIMESTAMP)) AS dia_num,
    CASE DAYOFWEEK(CAST(fecha AS TIMESTAMP))
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Lunes'
        WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miércoles'
        WHEN 5 THEN 'Jueves'
        WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sábado'
    END AS dia_semana,
    COUNT(*) AS ordenes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE estado = 'paid'
GROUP BY 1, 2
ORDER BY 1;

In [0]:
WITH ventas_mensuales AS (
    SELECT
        SUBSTRING(fecha, 1, 7)                                         AS mes,
        COUNT(*)                                                       AS ordenes,
        ROUND(SUM(
            TRY_CAST(precio_unitario AS DOUBLE) * TRY_CAST(cantidad AS INT)
        ), 2)                                                          AS ingresos_brutos,
        ROUND(SUM(TRY_CAST(comision AS DOUBLE)), 2)                    AS total_comisiones
    FROM iniciacion_deportiva.bronze.ventas_raw
    WHERE estado = 'paid'
    GROUP BY SUBSTRING(fecha, 1, 7)
),
con_metricas_avanzadas AS (
    SELECT
        mes,
        ordenes,
        ingresos_brutos,
        total_comisiones,
        -- Variación MoM (Mes a Mes)
        LAG(ingresos_brutos) OVER (ORDER BY mes)                      AS ingresos_mes_ant,
        -- Ingresos Acumulados (Running Total)
        ROUND(SUM(ingresos_brutos) OVER (ORDER BY mes), 2)            AS ingresos_acumulados,
        -- Órdenes Acumuladas
        SUM(ordenes) OVER (ORDER BY mes)                               AS ordenes_acumuladas
    FROM ventas_mensuales
)
SELECT
    mes,
    ordenes,
    ordenes_acumuladas,
    ingresos_brutos,
    ingresos_acumulados,
    total_comisiones,
    -- Cálculo de la variación porcentual MoM
    COALESCE(
        CAST(ROUND((ingresos_brutos - ingresos_mes_ant) * 100.0 / ingresos_mes_ant, 2) AS STRING), 
        'primer mes'
    ) AS var_vs_mes_anterior
FROM con_metricas_avanzadas
ORDER BY mes;

## Resumen Ejecutivo — Dashboard de Calidad
Una sola query que consolida todo lo encontrado en el EDA.

In [0]:
%sql
WITH metricas AS (
    SELECT
        COUNT(*)                                                                        AS total_registros,
        COUNT(DISTINCT id_venta)                                                        AS ventas_unicas,
        COUNT(DISTINCT id_producto)                                                     AS productos_unicos,
        COUNT(DISTINCT id_cliente)                                                      AS clientes_unicos,
        COUNT(DISTINCT categoria_id)                                                    AS categorias,
        MIN(SUBSTRING(fecha, 1, 10))                                                    AS fecha_inicio,
        MAX(SUBSTRING(fecha, 1, 10))                                                    AS fecha_fin,
        ROUND(COUNT(CASE WHEN estado = 'paid'      THEN 1 END) * 100.0 / COUNT(*), 2)  AS pct_pagadas,
        ROUND(COUNT(CASE WHEN estado = 'cancelled' THEN 1 END) * 100.0 / COUNT(*), 2)  AS pct_canceladas,
        COUNT(*) - COUNT(DISTINCT id_venta)                                             AS duplicados_en_bronze,
        COUNT(CASE WHEN precio_unitario IS NULL OR precio_unitario = '' THEN 1 END)     AS sin_precio,
        COUNT(CASE WHEN cantidad IS NULL OR cantidad = '' THEN 1 END)                   AS sin_cantidad
    FROM iniciacion_deportiva.bronze.ventas_raw
)
SELECT
    total_registros,
    ventas_unicas,
    duplicados_en_bronze,
    productos_unicos,
    clientes_unicos,
    categorias,
    fecha_inicio,
    fecha_fin,
    pct_pagadas    || '%' AS pct_pagadas,
    pct_canceladas || '%' AS pct_canceladas,
    sin_precio,
    sin_cantidad
FROM metricas;


### Decisiones Técnicas y conclusiones del EDA

**Bronze — ya implementado**
- Todo STRING excepto `ingested_at` — previene fallos si la API cambia tipos
- Duplicados esperados por diseño — cada venta aparece hasta ~8 veces (ventana 2hs, run cada 15 min)
- `_rescued_data` en NULL — schema estable
- **Canceladas ausentes por diseño de la API de Meli** — no es un bug del ETL. Las cancelaciones reales se manejan vía `/post-purchase/v1/claims`, un recurso separado de `/orders`. El `partially_refunded` sí aparece porque cambia el `status` de la orden directamente. Para capturar cancelaciones completas se necesita consumir la API de claims en una **v2 del pipeline**.

**Silver — a implementar**
- Deduplicar con `MERGE ON id_venta`
- MERGE con lógica de actualización de estado: `WHEN MATCHED AND source.estado != target.estado → UPDATE estado` (captura transiciones como `paid → partially_refunded`)
- `TRY_CAST` a tipos correctos: `precio_unitario` / `comision` → DOUBLE, `cantidad` → INT, `fecha` → TIMESTAMP
- Normalizar `titulo_producto` — 97 productos con múltiples títulos son edits menores del listing, no productos distintos. Quedarse con el más reciente:
```sql
ROW_NUMBER() OVER (PARTITION BY id_producto ORDER BY fecha DESC) = 1
```
- Campo derivado: `total_venta = precio_unitario * cantidad`
- Validaciones: descartar filas con `precio_unitario <= 0` o `cantidad <= 0`
- **`silver.categorias`** — tabla separada, un call a `/categories/{categoria_id}` por cada uno de los 180 IDs únicos. Se resuelve en Silver para que Gold solo haga JOIN sin tocar APIs externas.

**Gold — a diseñar**
- **Pedidos multi-ítem** — 1.109 casos confirmados. Meli devuelve una orden por producto, no por pedido. Agrupar por `(id_cliente, DATE_TRUNC('minute', fecha))` para reconstruir pedidos → ticket promedio real.
- **`dim_categoria`** — JOIN con `silver.categorias` para nombres legibles, sin llamar APIs desde Gold.
- **Outliers de precio** — son SKUs válidos del catálogo, no errores. No filtrar.
- **Limitación conocida** — canceladas históricamente ausentes. Documentar como nota en el dashboard. V2 del pipeline: consumir `/post-purchase/v1/claims`.